# HỆ THỐNG PHÁT HIỆN VÀ BÁM BẮT MỤC TIÊU UAV (ANCHOR-FREE CENTERNET RGB)
### Quy Trình Toàn Diện: Định Nghĩa Mô Hình -> Huấn Luyện / Tiếp Tục Huấn Luyện (Resume) -> Đánh Giá Benchmark -> Suy Luận Tác Chiến

Notebook này chứa toàn bộ quy trình hoàn chỉnh của mô hình Anti-UAV CenterNet thuần Visible RGB:
1. **Cấu hình & Tham số Toàn cục (Config)**
2. **Tiền xử lý & Nạp dữ liệu Temporal Triplet (Dataloader)**
3. **Kiến trúc Mạng Nơ-ron (ResNet50v2 + Coordinate Attention + P2-FPN + CenterNet Heads)**
4. **Hệ thống Hàm Mất mát Đa nhiệm (Gaussian Focal Loss + Masked L1 + NWD Loss)**
5. **Huấn luyện & Khôi phục Huấn luyện (Train & Resume từ Checkpoint gián đoạn)**
6. **Đánh giá Định lượng Benchmark (Precision, Recall, mAP@0.5, NWD-mAP, FPS)**
7. **Sử dụng Mô hình (Inference Ảnh/Video cục bộ hoặc URL)**

---
## 1. Cấu hình Môi trường Phần cứng và Tham số Toàn cục (Config)
Thiết lập cấp phát bộ nhớ GPU linh hoạt (Memory Growth) và định nghĩa các siêu tham số mạng nơ-ron.

In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import glob
import time
import math
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Cấu hình cấp phát bộ nhớ GPU linh hoạt tránh lỗi OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f">> GPU sẵn sàng: {tf.test.gpu_device_name()}")
else:
    print(">> Không phát hiện GPU, sẽ thực thi trên CPU.")

class Config:
    INPUT_HEIGHT = 640
    INPUT_WIDTH = 640
    INPUT_CHANNELS = 9
    INPUT_SHAPE = (640, 640, 9)

    ORIG_WIDTH = 1920.0
    ORIG_HEIGHT = 1080.0

    FEATURE_STRIDE = 8
    GRID_H = 80
    GRID_W = 80
    FPN_DIM = 128

    # Khởi tạo Logit Prior Bias giải quyết hiện tượng Dying ReLU
    SIZE_BIAS_INIT = -2.66       # ln(0.065 / (1 - 0.065))
    HEATMAP_BIAS_INIT = -4.595   # ln(0.01 / (1 - 0.01))

    # Trọng số cân bằng hàm mất mát
    LAMBDA_HEATMAP = 1.0
    LAMBDA_OFFSET = 1.0
    LAMBDA_SIZE = 1.0

    # Tham số huấn luyện
    DEFAULT_BATCH_SIZE = 8
    DEFAULT_LR = 1e-4
    DEFAULT_EPOCHS = 10

    # Ngưỡng tin cậy suy luận
    CONF_THRESH = 0.40
    TRACKER_ALPHA = 0.6
    TRACKER_MAX_MISSING = 15

    CHECKPOINT_DIR = "weights"
    OUTPUT_DIR = "outputs"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(">> Đã thiết lập Config thành công!")

---
## 2. Pipeline Tiền Xử Lý Dữ Liệu và Dataloader (Temporal Triplet)
* Ghép nối 3 khung hình liên tiếp $(I_{t-1}, I_t, I_{t+1})$ thành tensor không - thời gian 9 kênh.
* Xây dựng phân phối Gaussian Heatmap theo kích thước thực tế của mục tiêu drone trên lưới stride 8 ($80 	imes 80$).

In [ ]:
# -*- coding: utf-8 -*-
def decode_and_resize_img(file_path, channels=3, target_size=(640, 640)):
    """Đọc ảnh từ đĩa, chuyển đổi và thay đổi kích thước về target_size."""
    img_raw = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(img_raw, channels=channels)
    img = tf.image.resize(img, target_size, method="bilinear")
    return tf.cast(img, tf.float32) / 255.0

def random_horizontal_flip(tensor_multi_ch, bbox, exist):
    """Tăng cường dữ liệu bằng lật ngang ngẫu nhiên cả tensor 9 kênh và bounding box."""
    do_flip = tf.random.uniform([]) > 0.5
    if do_flip:
        tensor_multi_ch = tf.image.flip_left_right(tensor_multi_ch)
        xmin, ymin, xmax, ymax = bbox[0], bbox[1], bbox[2], bbox[3]
        new_xmin = 1.0 - xmax 
        new_xmax = 1.0 - xmin
        bbox = tf.stack([new_xmin, ymin, new_xmax, ymax])
    return tensor_multi_ch, bbox, exist

def build_preprocess_fn(voc_root, target_size=(640, 640), is_training=True):
    orig_w, orig_h = Config.ORIG_WIDTH, Config.ORIG_HEIGHT
    grid_h, grid_w = Config.GRID_H, Config.GRID_W

    def preprocess_sample(vis_tm1_rel, vis_t_rel, vis_tp1_rel, xmin, ymin, xmax, ymax, exist):
        p_vis_tm1 = tf.strings.join([voc_root, "/", vis_tm1_rel])
        p_vis_t   = tf.strings.join([voc_root, "/", vis_t_rel])
        p_vis_tp1 = tf.strings.join([voc_root, "/", vis_tp1_rel])

        img_tm1 = decode_and_resize_img(p_vis_tm1, 3, target_size)
        img_t   = decode_and_resize_img(p_vis_t,   3, target_size)
        img_tp1 = decode_and_resize_img(p_vis_tp1, 3, target_size)

        # Ghép thành tensor 9 kênh
        tensor_input = tf.concat([img_tm1, img_t, img_tp1], axis=-1)

        norm_xmin = tf.clip_by_value(tf.cast(xmin, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymin = tf.clip_by_value(tf.cast(ymin, tf.float32) / orig_h, 0.0, 1.0)
        norm_xmax = tf.clip_by_value(tf.cast(xmax, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymax = tf.clip_by_value(tf.cast(ymax, tf.float32) / orig_h, 0.0, 1.0)

        norm_box = tf.where(
            exist == 1,
            tf.stack([norm_xmin, norm_ymin, norm_xmax, norm_ymax]),
            tf.constant([0.0, 0.0, 0.0, 0.0], dtype=tf.float32)
        )

        if is_training:
            tensor_input, norm_box, exist = random_horizontal_flip(tensor_input, norm_box, exist)

        xs = tf.cast(tf.range(grid_w), tf.float32)
        ys = tf.cast(tf.range(grid_h), tf.float32)
        grid_x, grid_y = tf.meshgrid(xs, ys)

        cur_xmin, cur_ymin = norm_box[0], norm_box[1]
        cur_xmax, cur_ymax = norm_box[2], norm_box[3]

        cx = (cur_xmin + cur_xmax) / 2.0
        cy = (cur_ymin + cur_ymax) / 2.0
        bw = cur_xmax - cur_xmin
        bh = cur_ymax - cur_ymin

        px = cx * float(grid_w)
        py = cy * float(grid_h)
        ix = tf.clip_by_value(tf.cast(tf.math.floor(px), tf.int32), 0, grid_w - 1)
        iy = tf.clip_by_value(tf.cast(tf.math.floor(py), tf.int32), 0, grid_h - 1)

        dx = px - tf.cast(ix, tf.float32)
        dy = py - tf.cast(iy, tf.float32)

        sigma = tf.maximum(1.0, tf.sqrt(tf.maximum(bw * float(grid_w) * bh * float(grid_h), 1e-4)) / 3.0)
        dist_sq = tf.square(grid_x - tf.cast(ix, tf.float32)) + tf.square(grid_y - tf.cast(iy, tf.float32))
        gaussian = tf.exp(-dist_sq / (2.0 * tf.square(sigma)))
        gaussian = tf.expand_dims(gaussian, axis=-1)

        center_point_mask = tf.cast(
            tf.logical_and(tf.equal(tf.cast(grid_x, tf.int32), ix), tf.equal(tf.cast(grid_y, tf.int32), iy)),
            tf.float32
        )
        center_point_mask = tf.expand_dims(center_point_mask, axis=-1)
        center_point_mask = tf.where(exist == 1, center_point_mask, tf.zeros_like(center_point_mask))

        heatmap = tf.where(exist == 1, gaussian, tf.zeros_like(gaussian))
        offset_map = tf.concat([center_point_mask * dx, center_point_mask * dy], axis=-1)
        size_map = tf.concat([center_point_mask * bw, center_point_mask * bh], axis=-1)

        labels = {
            "heatmap": heatmap,
            "offset": offset_map,
            "size": size_map,
            "mask": center_point_mask,
            "raw_bbox": tf.cast(norm_box, tf.float32),
            "class_output": tf.cast(exist, tf.float32),
            "bbox_output": tf.cast(norm_box, tf.float32)
        }
        return tensor_input, labels

    return preprocess_sample

def create_temporal_dataset(csv_path, voc_root, batch_size=8, target_size=(640, 640), is_training=True, shuffle_buffer=1024):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file CSV tại: {csv_path}")
    df = pd.read_csv(csv_path)
    total_samples = len(df)

    ds = tf.data.Dataset.from_tensor_slices((
        df["vis_tm1"].values.astype(str),
        df["vis_t"].values.astype(str),
        df["vis_tp1"].values.astype(str),
        df["xmin"].values.astype(np.float32),
        df["ymin"].values.astype(np.float32),
        df["xmax"].values.astype(np.float32),
        df["ymax"].values.astype(np.float32),
        df["exist"].values.astype(np.int32)
    ))

    if is_training:
        ds = ds.shuffle(buffer_size=min(total_samples, shuffle_buffer), reshuffle_each_iteration=True)

    preprocess_fn = build_preprocess_fn(voc_root, target_size=target_size, is_training=is_training)
    ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=is_training)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds, total_samples

print(">> Pipeline nạp dữ liệu Dataloader đã sẵn sàng!")

---
## 3. Kiến Trúc Mạng Nơ-ron: ResNet50v2 + Coordinate Attention + P2-FPN + CenterNet Heads
* **Coordinate Attention:** Phân rã gom cụm 2D thành hai phép tính 1D dọc-ngang, mã hóa vị trí chính xác của drone nhỏ.
* **Anchor-Free CenterHead:** Phân tách thành 3 nhánh Heatmap, Offset, Size hồi quy trực tiếp điểm tâm mà không cần Anchor Boxes.

In [ ]:
# -*- coding: utf-8 -*-
class CoordinateAttention(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
    
    def build(self, input_shape):
        channels = input_shape[-1]
        reduced = max(8, channels // self.reduction)
        self.conv_shared = layers.Conv2D(reduced, kernel_size=1, strides=1, use_bias=True)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("relu")
        self.conv_h = layers.Conv2D(channels, kernel_size=1, activation="sigmoid", use_bias=True)
        self.conv_w = layers.Conv2D(channels, kernel_size=1, activation="sigmoid", use_bias=True)
        super().build(input_shape)
    
    def call(self, x):
        h = tf.shape(x)[1]
        w = tf.shape(x)[2]
        x_h = tf.reduce_mean(x, axis=2, keepdims=True)
        x_w = tf.reduce_mean(x, axis=1, keepdims=True)
        x_w_perm = tf.transpose(x_w, perm=[0, 2, 1, 3])

        concat = tf.concat([x_h, x_w_perm], axis=1)
        y = self.act(self.bn(self.conv_shared(concat)))

        y_h = y[:, :h, :, :]
        y_w = tf.transpose(y[:, h:, :, :], perm=[0, 2, 1, 3])
        att_h = self.conv_h(y_h)
        att_w = self.conv_w(y_w)
        return x * att_h * att_w

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config

def bottleneck_v2(x, filters, stride=1, projection=False, name=""):
    shortcut = x
    preact = layers.BatchNormalization(name=f"{name}_preact_bn")(x)
    preact = layers.Activation("relu", name=f"{name}_preact_relu")(preact)
    if projection:
        shortcut = layers.Conv2D(4 * filters, kernel_size=1, strides=stride, use_bias=False, name=f"{name}_proj_conv")(preact)

    x = layers.Conv2D(filters, kernel_size=1, strides=1, use_bias=False, name=f"{name}_1_conv")(preact)
    x = layers.BatchNormalization(name=f"{name}_1_bn")(x)
    x = layers.Activation("relu", name=f"{name}_1_relu")(x)

    x = layers.Conv2D(filters, kernel_size=3, strides=stride, padding="same", use_bias=False, name=f"{name}_2_conv")(x)
    x = layers.BatchNormalization(name=f"{name}_2_bn")(x)
    x = layers.Activation("relu", name=f"{name}_2_relu")(x)

    x = layers.Conv2D(4 * filters, kernel_size=1, strides=1, use_bias=False, name=f"{name}_3_conv")(x)
    return layers.Add(name=f"{name}_out")([shortcut, x])

def resnet50v2_backbone(input_tensor):
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding="same", use_bias=False, name="stem_conv")(input_tensor)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding="same", name="stem_pool")(x)

    # C2 (stride 4)
    x = bottleneck_v2(x, 64, stride=1, projection=True, name="c2_b1")
    x = bottleneck_v2(x, 64, stride=1, name="c2_b2")
    c2 = bottleneck_v2(x, 64, stride=1, name="c2_b3")

    # C3 (stride 8)
    x = bottleneck_v2(c2, 128, stride=2, projection=True, name="c3_b1")
    for i in range(2, 5): x = bottleneck_v2(x, 128, stride=1, name=f"c3_b{i}")
    c3 = x

    # C4 (stride 16)
    x = bottleneck_v2(c3, 256, stride=2, projection=True, name="c4_b1")
    for i in range(2, 7): x = bottleneck_v2(x, 256, stride=1, name=f"c4_b{i}")
    c4 = x

    # C5 (stride 32)
    x = bottleneck_v2(c4, 512, stride=2, projection=True, name="c5_b1")
    x = bottleneck_v2(x, 512, stride=1, name="c5_b2")
    c5 = bottleneck_v2(x, 512, stride=1, name="c5_b3")
    return c2, c3, c4, c5

def fpn(c2, c3, c4, c5, fpn_dim=128):
    lat_c5 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c5")(c5)
    lat_c4 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c4")(c4)
    lat_c3 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c3")(c3)
    lat_c2 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c2")(c2)

    p5 = lat_c5
    p4 = layers.Add(name="fpn_add_p4")([layers.UpSampling2D(size=2, name="fpn_up_p5")(p5), lat_c4])
    p3 = layers.Add(name="fpn_add_p3")([layers.UpSampling2D(size=2, name="fpn_up_p4")(p4), lat_c3])
    p2 = layers.Add(name="fpn_add_p2")([layers.UpSampling2D(size=2, name="fpn_up_p3")(p3), lat_c2])

    p5 = CoordinateAttention(reduction=16, name="fpn_ca_p5")(layers.Conv2D(fpn_dim, 3, padding="same", name="fpn_smooth_p5")(p5))
    p4 = CoordinateAttention(reduction=16, name="fpn_ca_p4")(layers.Conv2D(fpn_dim, 3, padding="same", name="fpn_smooth_p4")(p4))
    p3 = CoordinateAttention(reduction=16, name="fpn_ca_p3")(layers.Conv2D(fpn_dim, 3, padding="same", name="fpn_smooth_p3")(p3))
    p2 = CoordinateAttention(reduction=16, name="fpn_ca_p2")(layers.Conv2D(fpn_dim, 3, padding="same", name="fpn_smooth_p2")(p2))
    return p2, p3, p4, p5

class AnchorFreeCenterHead(layers.Layer):
    def __init__(self, fpn_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.fpn_dim = fpn_dim

    def build(self, input_shape):
        self.hm_conv1 = layers.Conv2D(128, 3, padding="same", activation="relu", name="hm_conv1")
        self.hm_bn1 = layers.BatchNormalization(name="hm_bn1")
        self.hm_out = layers.Conv2D(1, 1, padding="same", activation="sigmoid",
                                    bias_initializer=tf.keras.initializers.Constant(Config.HEATMAP_BIAS_INIT),
                                    dtype="float32", name="heatmap")

        self.off_conv1 = layers.Conv2D(64, 3, padding="same", activation="relu", name="off_conv1")
        self.off_bn1 = layers.BatchNormalization(name="off_bn1")
        self.off_out = layers.Conv2D(2, 1, padding="same", dtype="float32", name="offset")

        self.size_conv1 = layers.Conv2D(64, 3, padding="same", activation="relu", name="size_conv1")
        self.size_bn1 = layers.BatchNormalization(name="size_bn1")
        self.size_out = layers.Conv2D(2, 1, padding="same", activation="sigmoid",
                                      bias_initializer=tf.keras.initializers.Constant(Config.SIZE_BIAS_INIT),
                                      dtype="float32", name="size")
        super().build(input_shape)

    def call(self, x):
        pred_hm = self.hm_out(self.hm_bn1(self.hm_conv1(x)))
        pred_off = self.off_out(self.off_bn1(self.off_conv1(x)))
        pred_size = self.size_out(self.size_bn1(self.size_conv1(x)))
        return {"heatmap": pred_hm, "offset": pred_off, "size": pred_size}

def build_detection_model(input_shape=(640, 640, 9), fpn_dim=128, name="ResNet50v2_FPN"):
    inputs = layers.Input(shape=input_shape, name="temporal_input")
    c2, c3, c4, c5 = resnet50v2_backbone(inputs)
    p2, p3, p4, p5 = fpn(c2, c3, c4, c5, fpn_dim=fpn_dim)

    p2_down = layers.MaxPool2D(pool_size=2, name="fuse_p2_down")(p2)
    p3_same = p3
    p4_up = layers.UpSampling2D(size=2, name="fuse_p4_up")(p4)
    p5_up = layers.UpSampling2D(size=4, name="fuse_p5_up")(p5)

    fused = layers.Concatenate(axis=-1, name="fuse_concat")([p2_down, p3_same, p4_up, p5_up])
    fused = layers.BatchNormalization(name="fuse_bn")(
        layers.Conv2D(fpn_dim, kernel_size=3, padding="same", activation="relu", name="fuse_conv")(fused)
    )
    outputs = AnchorFreeCenterHead(fpn_dim=fpn_dim, name="anchor_free_head")(fused)
    return Model(inputs=inputs, outputs=outputs, name=name)

def decode_detections(heatmap, offset, size, default_w=0.065, default_h=0.070, min_size=0.015):
    hmax = tf.nn.max_pool2d(heatmap, ksize=3, strides=1, padding="SAME")
    keep = tf.cast(tf.equal(heatmap, hmax), tf.float32)
    peak_heatmap = heatmap * keep

    B = tf.shape(heatmap)[0]
    H, W = tf.shape(heatmap)[1], tf.shape(heatmap)[2]

    flat_peaks = tf.reshape(peak_heatmap, [B, H * W])
    top_scores, top_indices = tf.math.top_k(flat_peaks, k=1)
    score = top_scores[:, 0]
    idx = top_indices[:, 0]

    grid_x = tf.cast(idx % W, tf.float32)
    grid_y = tf.cast(idx // W, tf.float32)

    flat_offset = tf.reshape(offset, [B, H * W, 2])
    flat_size = tf.reshape(size, [B, H * W, 2])

    batch_indices = tf.range(B, dtype=tf.int32)
    gather_idx = tf.stack([batch_indices, idx], axis=-1)

    peak_offset = tf.gather_nd(flat_offset, gather_idx)
    peak_size = tf.gather_nd(flat_size, gather_idx)

    cx = (grid_x + peak_offset[:, 0]) / tf.cast(W, tf.float32)
    cy = (grid_y + peak_offset[:, 1]) / tf.cast(H, tf.float32)

    bw = tf.where(peak_size[:, 0] < min_size, tf.constant(default_w, dtype=tf.float32), peak_size[:, 0])
    bh = tf.where(peak_size[:, 1] < min_size, tf.constant(default_h, dtype=tf.float32), peak_size[:, 1])

    xmin = tf.clip_by_value(cx - bw / 2.0, 0.0, 1.0)
    ymin = tf.clip_by_value(cy - bh / 2.0, 0.0, 1.0)
    xmax = tf.clip_by_value(cx + bw / 2.0, 0.0, 1.0)
    ymax = tf.clip_by_value(cy + bh / 2.0, 0.0, 1.0)
    bboxes = tf.stack([xmin, ymin, xmax, ymax], axis=-1)
    return score, bboxes

decode_detection = decode_detections
print(">> Kiến trúc mô hình CenterNet RGB đã sẵn sàng!")

---
## 4. Hệ Thống Hàm Mất Mát Đa Nhiệm (Loss Functions)
* **Gaussian Focal Loss:** Xử lý mất cân bằng nghiêm trọng giữa điểm tâm và nền trời.
* **Masked L1 Loss:** Huấn luyện sai lệch dưới pixel (sub-pixel offset).
* **Normalized Wasserstein Distance (NWD):** Đo khoảng cách phân phối Gaussian 2D, giải quyết triệt để vấn đề mất gradient của IoU với micro-drone.

In [ ]:
# -*- coding: utf-8 -*-
def gaussian_focal_loss(y_true, y_pred, alpha=2.0, beta=4.0):
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    pos_mask = tf.cast(tf.equal(y_true, 1.0), tf.float32)
    neg_mask = tf.cast(tf.less(y_true, 1.0), tf.float32)

    pos_loss = -tf.pow(1.0 - y_pred, alpha) * tf.math.log(y_pred) * pos_mask
    neg_loss = -tf.pow(1.0 - y_true, beta) * tf.pow(y_pred, alpha) * tf.math.log(1.0 - y_pred) * neg_mask
    num_pos = tf.reduce_sum(pos_mask)
    return (tf.reduce_sum(pos_loss) + tf.reduce_sum(neg_loss)) / tf.maximum(num_pos, 1.0)

def masked_l1_loss(y_true, y_pred, mask):
    diff = tf.abs(y_true - y_pred) * mask
    num_pos = tf.reduce_sum(mask)
    return tf.reduce_sum(diff) / tf.maximum(num_pos, 1.0)

def compute_nwd_size_loss(y_true_sz, y_pred_sz, mask, c_norm=0.10):
    w_true, h_true = y_true_sz[..., 0:1], y_true_sz[..., 1:2]
    w_pred, h_pred = y_pred_sz[..., 0:1], y_pred_sz[..., 1:2]
    w2_dist = tf.square(w_true - w_pred) + tf.square(h_true - h_pred)
    w_dist = tf.sqrt(tf.maximum(w2_dist, 1e-7))
    nwd_sim = tf.exp(-w_dist / c_norm)
    nwd_loss = (1.0 - nwd_sim) * mask
    num_pos = tf.reduce_sum(mask)
    return tf.reduce_sum(nwd_loss) / tf.maximum(num_pos, 1.0)

def size_regression_loss(y_true_sz, y_pred_sz, mask, c_norm=0.10):
    l1 = masked_l1_loss(y_true_sz, y_pred_sz, mask)
    nwd = compute_nwd_size_loss(y_true_sz, y_pred_sz, mask, c_norm)
    return l1 * 5.0 + nwd

def compute_combined_loss(y_true_dict, y_pred_dict, lambda_hm=1.0, lambda_off=1.0, lambda_size=1.0):
    y_true_hm, y_pred_hm = tf.cast(y_true_dict["heatmap"], tf.float32), tf.cast(y_pred_dict["heatmap"], tf.float32)
    y_true_off, y_pred_off = tf.cast(y_true_dict["offset"], tf.float32), tf.cast(y_pred_dict["offset"], tf.float32)
    y_true_size, y_pred_size = tf.cast(y_true_dict["size"], tf.float32), tf.cast(y_pred_dict["size"], tf.float32)
    mask = tf.cast(y_true_dict["mask"], tf.float32)

    loss_hm = gaussian_focal_loss(y_true_hm, y_pred_hm)
    loss_off = masked_l1_loss(y_true_off, y_pred_off, mask)
    loss_size = size_regression_loss(y_true_size, y_pred_size, mask)

    total_loss = lambda_hm * loss_hm + lambda_off * loss_off + lambda_size * loss_size
    return total_loss, loss_hm, (lambda_off * loss_off + lambda_size * loss_size)

print(">> Hệ thống hàm mất mát đã sẵn sàng")

---
## 5. Huấn Luyện Mô Hình Từ Đầu & Khôi Phục Huấn Luyện (Training Pipeline)
* **Huấn luyện từ đầu (Train from scratch)**: Khởi tạo mô hình mới với các trọng số ban đầu, thực thi tối ưu hóa theo gradient qua từng Epoch từ Epoch 1.
* **Khôi phục huấn luyện (Resume Training)**: Hỗ trợ nạp checkpoint đã lưu (`latest_uav_model.keras` hoặc `best_uav_model.keras`) nếu quá trình huấn luyện bị ngắt quãng giữa chừng khi người dùng cấu hình `resume=True`.
* Toàn bộ lịch sử tổn thất sẽ được tích lũy trực tiếp vào biến `history` trong bộ nhớ trong suốt phiên huấn luyện.

In [ ]:
# -*- coding: utf-8 -*-
def run_training(voc_root, train_csv, val_csv, epochs=Config.DEFAULT_EPOCHS,
                 batch_size=Config.DEFAULT_BATCH_SIZE, lr=Config.DEFAULT_LR,
                 checkpoint_dir=Config.CHECKPOINT_DIR, resume=False):
    """
    Ham thuc thi huan luyen mo hinh tu dau hoac tiep tuc tu checkpoint neu bi gian doan.
    - resume=False (Mac dinh): Khoi tao mo hinh moi va huan luyen tu Epoch 1.
    - resume=True: Khoi phuc trong so tu checkpoint va tiep tuc cac epoch con lai.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    history_csv = os.path.join(checkpoint_dir, "training_history.csv")
    latest_ckpt = os.path.join(checkpoint_dir, "latest_uav_model.keras")
    best_ckpt   = os.path.join(checkpoint_dir, "best_uav_model.keras")

    # 1. Nap Dataset
    print(">> Nap tap du lieu Train & Val...")
    train_ds, n_train = create_temporal_dataset(train_csv, voc_root, batch_size=batch_size, is_training=True)
    val_ds, n_val     = create_temporal_dataset(val_csv,   voc_root, batch_size=batch_size, is_training=False)

    train_steps = n_train // batch_size
    val_steps   = n_val // batch_size
    print(f"   Train samples: {n_train} ({train_steps} steps) | Val samples: {n_val} ({val_steps} steps)")

    # 2. Xay dung kien truc mo hinh
    model = build_detection_model(input_shape=Config.INPUT_SHAPE, fpn_dim=Config.FPN_DIM)

    # 3. Kiem tra khoi phuc tu Checkpoint (chi khi resume=True)
    start_epoch = 1
    best_val_loss = float("inf")
    history_records = []

    if resume:
        ckpt_to_load = None
        if os.path.exists(latest_ckpt):
            ckpt_to_load = latest_ckpt
        elif os.path.exists(best_ckpt):
            ckpt_to_load = best_ckpt

        if ckpt_to_load:
            try:
                try:
                    loaded = tf.keras.models.load_model(
                        ckpt_to_load,
                        custom_objects={'CoordinateAttention': CoordinateAttention, 'AnchorFreeCenterHead': AnchorFreeCenterHead},
                        compile=False
                    )
                    model.set_weights(loaded.get_weights())
                except Exception:
                    model.load_weights(ckpt_to_load)
                print(f">> [RESUME] Da khoi phuc thanh cong trong so tu: {ckpt_to_load}")
            except Exception as e:
                print(f">> [CANH BAO] Khong the nap checkpoint: {e}. Tien hanh huan luyen tu dau.")

    # 4. Optimizer AdamW + CosineDecay
    decay_steps = max(1, epochs * train_steps)
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=lr, decay_steps=decay_steps, alpha=0.01)
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

    @tf.function
    def train_step(x, y_true):
        with tf.GradientTape() as tape:
            y_pred = model(x, training=True)
            tl, hl, sl = compute_combined_loss(y_true, y_pred)
        grads = tape.gradient(tl, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 10.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return tl, hl, sl

    @tf.function
    def val_step(x, y_true):
        y_pred = model(x, training=False)
        return compute_combined_loss(y_true, y_pred)

    if start_epoch > epochs:
        print(f">> Qua trinh huan luyen da hoan tat qua {epochs} Epochs!")
        return model, history_records

    print(f">> Bat dau huan luyen tu Epoch {start_epoch} den Epoch {epochs}...")
    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        print(f"--- EPOCH {epoch:02d}/{epochs:02d} ---")

        train_tl, train_hl, train_sl = 0.0, 0.0, 0.0
        pbar_tr = tqdm(total=train_steps, desc="Train", unit="batch")
        for step, (x_b, y_b) in enumerate(train_ds):
            if step >= train_steps: break
            tl, hl, sl = train_step(x_b, y_b)
            train_tl += float(tl); train_hl += float(hl); train_sl += float(sl)
            pbar_tr.set_postfix({"loss": f"{tl:.4f}", "hm": f"{hl:.4f}"})
            pbar_tr.update(1)
        pbar_tr.close()

        train_tl /= max(1, train_steps); train_hl /= max(1, train_steps); train_sl /= max(1, train_steps)

        val_tl, val_hl, val_sl = 0.0, 0.0, 0.0
        pbar_vl = tqdm(total=val_steps, desc="Val", unit="batch")
        for step, (x_b, y_b) in enumerate(val_ds):
            if step >= val_steps: break
            tl, hl, sl = val_step(x_b, y_b)
            val_tl += float(tl); val_hl += float(hl); val_sl += float(sl)
            pbar_vl.set_postfix({"val_loss": f"{tl:.4f}"})
            pbar_vl.update(1)
        pbar_vl.close()

        val_tl /= max(1, val_steps); val_hl /= max(1, val_steps); val_sl /= max(1, val_steps)
        elapsed = (time.time() - t0) / 60.0

        print(f"Epoch {epoch:02d}/{epochs:02d} ({elapsed:.1f} phut):")
        print(f"   TRAIN -> Total Loss: {train_tl:.4f} | Heatmap: {train_hl:.4f} | Size: {train_sl:.4f}")
        print(f"   VAL   -> Total Loss: {val_tl:.4f} | Heatmap: {val_hl:.4f} | Size: {val_sl:.4f}")

        # Luu checkpoint latest
        model.save_weights(latest_ckpt)

        # Luu checkpoint best
        if val_tl < best_val_loss:
            best_val_loss = val_tl
            model.save_weights(best_ckpt)
            print(f">> [KY LUC MOI] Da luu mo hinh tot nhat voi Val Loss: {best_val_loss:.4f} tai: {best_ckpt}")

        history_records.append({
            "epoch": epoch,
            "train_loss": train_tl, "train_hm_loss": train_hl, "train_size_loss": train_sl,
            "val_loss": val_tl,     "val_hm_loss": val_hl,     "val_size_loss": val_sl,
            "time_minutes": elapsed
        })
        pd.DataFrame(history_records).to_csv(history_csv, index=False)

    print(">> Huan luyen hoan thanh!")
    return model, history_records

# THUC THI HUAN LUYEN TU DAU (TRAIN FROM SCRATCH)
# Mac dinh resume=False de huan luyen mo hinh moi hoan toan tu Epoch 1
TRAIN_DATASET_EXISTS = os.path.exists(Config.TRAIN_CSV) and os.path.exists(Config.VAL_CSV)

if TRAIN_DATASET_EXISTS:
    print(">> Phat hien tap du lieu hop le. Bat dau huan luyen tu dau...")
    model, history = run_training(
        voc_root=Config.VOC_ROOT,
        train_csv=Config.TRAIN_CSV,
        val_csv=Config.VAL_CSV,
        epochs=Config.DEFAULT_EPOCHS,
        batch_size=Config.DEFAULT_BATCH_SIZE,
        lr=Config.DEFAULT_LR,
        checkpoint_dir=Config.CHECKPOINT_DIR,
        resume=False
    )
else:
    print(">> [CHU Y] Tap du lieu VOC huan luyen chua duoc dat tai duong dan cau hinh.")
    print(">> Khoi tao san kien truc mo hinh moi trong bo nho san sang cho huan luyen hoac danh gia:")
    model = build_detection_model(input_shape=Config.INPUT_SHAPE, fpn_dim=Config.FPN_DIM)
    model.summary()
    history = []

---
## 6. Đồ Thị Hội Tụ Huấn Luyện (Loss Curves)
Trực quan hóa tiến trình suy giảm của hàm mất mát tổng hợp và các nhánh thành phần trực tiếp từ kết quả huấn luyện mới sinh ra của phiên chạy hiện tại (không nạp log lịch sử cũ).

In [ ]:
# -*- coding: utf-8 -*-
# Truc quan hoa do thi hoi tu truc tiep tu bien history cua phien huan luyen nay
if 'history' in locals() and history and len(history) > 0:
    df_hist = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=200)

    axes[0].plot(df_hist["epoch"], df_hist["train_loss"], "b-o", label="Train Loss", linewidth=2)
    axes[0].plot(df_hist["epoch"], df_hist["val_loss"], "r--s", label="Val Loss", linewidth=2)
    axes[0].set_title("Hoi Tu Ham Mat Mat Tong Hop (Total Loss)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch", fontsize=11, fontweight="bold")
    axes[0].set_ylabel("Total Loss", fontsize=11, fontweight="bold")
    axes[0].legend(fontsize=10)
    axes[0].grid(True, linestyle=":", alpha=0.6)

    axes[1].plot(df_hist["epoch"], df_hist["train_hm_loss"], "g-^", label="Train Heatmap Loss", linewidth=2)
    axes[1].plot(df_hist["epoch"], df_hist["val_hm_loss"], "m--v", label="Val Heatmap Loss", linewidth=2)
    axes[1].plot(df_hist["epoch"], df_hist["train_size_loss"], "c:", label="Train Size/Offset", linewidth=2)
    axes[1].plot(df_hist["epoch"], df_hist["val_size_loss"], "y-.", label="Val Size/Offset", linewidth=2)
    axes[1].set_title("Chi Tiet Thanh Phan Heatmap & Size Loss", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch", fontsize=11, fontweight="bold")
    axes[1].set_ylabel("Component Loss", fontsize=11, fontweight="bold")
    axes[1].legend(fontsize=10)
    axes[1].grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()
    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/loss_curve.png", bbox_inches="tight")
    plt.show()
else:
    print(">> [THONG BAO] Chua co du lieu lich su huan luyen trong phien chay hien tai.")
    print(">> Do thi hoi tu se duoc tu dong ve sau khi chay qua trinh huan luyen tai Muc 5.")

---
## 7. Đánh Giá Định Lượng Mô Hình (Evaluation Benchmark)
Tính toán các chỉ số chuẩn thị giác máy tính: IoU, NWD, Precision, Recall, F1-Score, mAP@0.5, mAP@0.75, NWD-mAP và tốc độ suy luận (FPS).

In [ ]:
# -*- coding: utf-8 -*-
def compute_iou(b1, b2):
    x1 = np.maximum(b1[0], b2[0]); y1 = np.maximum(b1[1], b2[1])
    x2 = np.minimum(b1[2], b2[2]); y2 = np.minimum(b1[3], b2[3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    a1 = np.maximum(0.0, b1[2] - b1[0]) * np.maximum(0.0, b1[3] - b1[1])
    a2 = np.maximum(0.0, b2[2] - b2[0]) * np.maximum(0.0, b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 1e-7 else 0.0

def compute_nwd(b1, b2, c_norm=0.02):
    cx1, cy1 = (b1[0] + b1[2]) / 2.0, (b1[1] + b1[3]) / 2.0
    w1, h1   = np.maximum(b1[2] - b1[0], 1e-4), np.maximum(b1[3] - b1[1], 1e-4)
    cx2, cy2 = (b2[0] + b2[2]) / 2.0, (b2[1] + b2[3]) / 2.0
    w2, h2   = np.maximum(b2[2] - b2[0], 1e-4), np.maximum(b2[3] - b2[1], 1e-4)
    d2 = (cx1 - cx2)**2 + (cy1 - cy2)**2 + ((w1 - w2)**2 + (h1 - h2)**2) / 4.0
    return float(np.exp(-np.sqrt(np.maximum(d2, 1e-7)) / c_norm))

def compute_ap(y_true, y_scores, matches):
    idx = np.argsort(-y_scores)
    matches = matches[idx]
    y_true = y_true[idx]
    n_pos = np.sum(y_true)
    if n_pos == 0: return 0.0, np.zeros(1), np.zeros(1)
    tp_cumsum = np.cumsum(matches)
    fp_cumsum = np.cumsum(~matches)
    recalls = tp_cumsum / n_pos
    precisions = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1e-7)
    ap = 0.0
    for r in np.linspace(0.0, 1.0, 101):
        p = precisions[recalls >= r]
        ap += np.max(p) if len(p) > 0 else 0.0
    return ap / 101.0, recalls, precisions

def evaluate_model_benchmark(eval_model, eval_csv, voc_root, batch_size=8, conf_thresh=0.40, max_samples=None):
    """
    Danh gia hieu nang dinh luong mo hinh tren tap kiem thu: AP@0.5, AP@0.75, NWD-AP@0.5 va FPS.
    """
    if not os.path.exists(eval_csv):
        print(f">> [CHU Y] Khong tim thay tep CSV danh gia tai: {eval_csv}")
        return None

    eval_ds, n_eval = create_temporal_dataset(eval_csv, voc_root, batch_size=batch_size, is_training=False)
    if max_samples and max_samples < n_eval:
        n_eval = max_samples

    total_steps = int(np.ceil(n_eval / batch_size))
    y_true_cls, y_scores, matches_50, matches_75, matches_nwd = [], [], [], [], []
    center_errors_px = []
    total_time, total_frames = 0.0, 0

    @tf.function
    def pred_step(x):
        preds = eval_model(x, training=False)
        return decode_detections(preds["heatmap"], preds["offset"], preds["size"])

    print(f">> Bat dau danh gia dinh luong ({n_eval} mau kiem thu)...")
    for step, (x_b, y_b) in enumerate(tqdm(eval_ds, total=total_steps, desc="Benchmark")):
        if step >= total_steps: break
        t_start = time.time()
        scores, bboxes = pred_step(x_b)
        elapsed = time.time() - t_start
        total_time += elapsed
        total_frames += x_b.shape[0]

        b_exist = y_b["class_output"].numpy().astype(bool)
        b_gt_boxes = y_b["bbox_output"].numpy()
        b_pred_scores = scores.numpy()
        b_pred_boxes = bboxes.numpy()

        for gt_exist, gt_box, score, pred_box in zip(b_exist, b_gt_boxes, b_pred_scores, b_pred_boxes):
            y_true_cls.append(gt_exist)
            y_scores.append(score)
            if gt_exist:
                iou = float(compute_iou(gt_box, pred_box))
                nwd = float(compute_nwd(gt_box, pred_box))
                matches_50.append(iou >= 0.50)
                matches_75.append(iou >= 0.75)
                matches_nwd.append(nwd >= 0.50)
                if score >= conf_thresh:
                    cx_gt = (gt_box[0] + gt_box[2]) / 2.0 * 1920.0
                    cy_gt = (gt_box[1] + gt_box[3]) / 2.0 * 1080.0
                    cx_pr = (pred_box[0] + pred_box[2]) / 2.0 * 1920.0
                    cy_pr = (pred_box[1] + pred_box[3]) / 2.0 * 1080.0
                    center_errors_px.append(np.sqrt((cx_gt - cx_pr)**2 + (cy_gt - cy_pr)**2))
            else:
                matches_50.append(False)
                matches_75.append(False)
                matches_nwd.append(False)

    y_true_cls = np.array(y_true_cls)
    y_scores = np.array(y_scores)
    matches_50 = np.array(matches_50)
    matches_75 = np.array(matches_75)
    matches_nwd = np.array(matches_nwd)

    ap_50, _, _ = compute_ap(y_true_cls, y_scores, matches_50)
    ap_75, _, _ = compute_ap(y_true_cls, y_scores, matches_75)
    ap_nwd, _, _ = compute_ap(y_true_cls, y_scores, matches_nwd)
    mean_c_err = float(np.mean(center_errors_px)) if len(center_errors_px) > 0 else 0.0
    fps = total_frames / max(total_time, 1e-5)

    print("\n--- KET QUA DANH GIA BENCHMARK ---")
    print(f"   AP@0.50     : {ap_50 * 100:.2f}%")
    print(f"   AP@0.75     : {ap_75 * 100:.2f}%")
    print(f"   NWD-AP@0.50 : {ap_nwd * 100:.2f}%")
    print(f"   Center Error: {mean_c_err:.2f} pixels")
    print(f"   Toc do suy luan: {fps:.1f} FPS")
    return {"AP50": ap_50, "AP75": ap_75, "NWD_AP50": ap_nwd, "Center_Error": mean_c_err, "FPS": fps}

# Thuc thi danh gia neu co tap kiem thu
if os.path.exists(Config.VAL_CSV) and os.path.exists(Config.VOC_ROOT):
    eval_results = evaluate_model_benchmark(model, Config.VAL_CSV, Config.VOC_ROOT)
else:
    print(">> [THONG BAO] Ham danh gia evaluate_model_benchmark() da san sang de thuc thi khi co tap kiem thu.")

---
## 8. Bộ Lọc Quán Tính Quỹ Đạo (Trajectory EMA) & Kết Xuất HUD Tác Chiến

In [ ]:
# -*- coding: utf-8 -*-
class TrajectoryEMAFilter:
    """Bộ lọc trung bình động hàm mũ quỹ đạo hỗ trợ bám bắt mượt mà và dự báo vị trí khi mục tiêu bị che khuất."""
    def __init__(self, alpha=0.6, max_missing=15):
        self.alpha = alpha
        self.max_missing = max_missing
        self.smooth_box = None
        self.velocity = np.array([0.0, 0.0])
        self.missing_count = 0
        self.is_active = False

    def update(self, detected_box, is_detected):
        if is_detected and detected_box is not None:
            cur_box = np.array(detected_box, dtype=np.float32)
            cur_cx = (cur_box[0] + cur_box[2]) / 2.0
            cur_cy = (cur_box[1] + cur_box[3]) / 2.0

            if self.smooth_box is None:
                self.smooth_box = cur_box
                self.velocity = np.array([0.0, 0.0])
            else:
                prev_cx = (self.smooth_box[0] + self.smooth_box[2]) / 2.0
                prev_cy = (self.smooth_box[1] + self.smooth_box[3]) / 2.0
                self.velocity = self.alpha * self.velocity + 0.4 * np.array([cur_cx - prev_cx, cur_cy - prev_cy])
                self.smooth_box = self.alpha * cur_box + (1.0 - self.alpha) * self.smooth_box

            self.missing_count = 0
            self.is_active = True
            return self.smooth_box.copy(), "DETECTED", float(np.linalg.norm(self.velocity))
        else:
            if self.is_active and self.missing_count < self.max_missing:
                self.missing_count += 1
                self.smooth_box[0] += self.velocity[0]
                self.smooth_box[1] += self.velocity[1]
                self.smooth_box[2] += self.velocity[0]
                self.smooth_box[3] += self.velocity[1]
                return self.smooth_box.copy(), "OCCLUDED", float(np.linalg.norm(self.velocity))
            else:
                self.is_active = False
                self.smooth_box = None
                return None, "LOST", 0.0

def draw_hud_reticle(img, bbox, conf=1.0, speed=0.0, state="DETECTED"):
    """Vẽ tâm ngắm chiến thuật, bounding box viền mỏng và thông số viễn thám lên khung hình."""
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    w, h = max(1, x2 - x1), max(1, y2 - y1)
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

    color = (0, 255, 0) if state == "DETECTED" else ((0, 215, 255) if state == "OCCLUDED" else (150, 150, 150))
    line_len = max(8, min(w, h) // 3)
    t = 2

    # 4 góc ngắm Bracket
    cv2.line(img, (x1, y1), (x1 + line_len, y1), color, t)
    cv2.line(img, (x1, y1), (x1, y1 + line_len), color, t)
    cv2.line(img, (x2, y1), (x2 - line_len, y1), color, t)
    cv2.line(img, (x2, y1), (x2, y1 + line_len), color, t)
    cv2.line(img, (x1, y2), (x1 + line_len, y2), color, t)
    cv2.line(img, (x1, y2), (x1, y2 - line_len), color, t)
    cv2.line(img, (x2, y2), (x2 - line_len, y2), color, t)
    cv2.line(img, (x2, y2), (x2, y2 - line_len), color, t)

    # Khung box viền mảnh (thickness=1) và tâm đối tượng
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
    cv2.circle(img, (cx, cy), 2, color, -1)

    infor_str = f"DRONE {conf*100:.1f}% | {w}x{h}px | v={speed:.1f}px/f"
    (fw, fh), _ = cv2.getTextSize(infor_str, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    badge_y1 = max(0, y1 - fh - 8)
    badge_y2 = badge_y1 + fh + 6
    cv2.rectangle(img, (x1, badge_y1), (x1 + fw + 8, badge_y2), (20, 20, 20), -1)
    cv2.rectangle(img, (x1, badge_y1), (x1 + fw + 8, badge_y2), color, 1)
    cv2.putText(img, infor_str, (x1 + 4, max(12, y1 - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

def create_tactical_canvas(frame_v, title="ANTI-UAV RGB", state="DETECTED", frame_info=""):
    out_w, out_h = 960, 540
    canvas_w = out_w
    canvas_h = out_h + 60

    disp_v = cv2.resize(frame_v, (out_w, out_h))

    # Thanh Header HUD phía trên
    header = np.zeros((60, canvas_w, 3), dtype=np.uint8)
    header[:, :] = (20, 25, 30)

    # Tiêu đề chỉ ghi "ANTI-UAV RGB"
    cv2.putText(header, "ANTI-UAV RGB", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
    if frame_info:
        cv2.putText(header, frame_info, (20, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)

    stat_color = (0, 255, 0) if state == "DETECTED" else ((0, 215, 255) if state == "OCCLUDED" else (100, 100, 100))
    cv2.putText(header, f"STATUS: [{state}]", (canvas_w - 280, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, stat_color, 2)

    return np.vstack([header, disp_v])

print(">> Công cụ đã sẵn sàng!")

---
## 9. Thực Thi Suy Luận & Render Tác Chiến (Tệp Cục Bộ hoặc Đường Dẫn URL)
Hướng dẫn sử dụng:
* **Chạy video/ảnh có sẵn trong máy:** Điền đường dẫn tệp vào biến `MEDIA_SOURCE` (ví dụ: `../assets/train_002_visible.mp4` hoặc `../assets/fig_visual_test_samples.png`).
* **Chạy qua liên kết URL từ xa:** Dán trực tiếp địa chỉ URL (ví dụ: `https://example.com/drone_test.mp4`), hệ thống sẽ tự động tải về và chạy suy luận.

In [ ]:
# -*- coding: utf-8 -*-
import urllib.request
import tempfile
from IPython.display import HTML, display
from base64 import b64encode

# CAU HINH DAU VAO SUY LUAN (Ho tro ca duong dan file cuc bo va link URL)
MEDIA_SOURCE = "../assets/train_002_visible.mp4"  # Hoac dan link URL: "https://.../video.mp4"
CONFIDENCE_THRESHOLD = 0.40
MAX_FRAMES_TO_RENDER = 200

# 1. Kiem tra mo hinh san co trong bo nho hoac nap tu Checkpoint
infer_model = None
if 'model' in locals() and model is not None:
    infer_model = model
    print(">> Su dung mo hinh san co trong bo nho de suy luan.")
else:
    ckpt_candidates = [
        "weights/best_uav_model.keras",
        "../weights/best_uav_model.keras",
        "weights/latest_uav_model.keras",
        "/kaggle/input/datasets/namnguyen171006/uav-checkpoint/best_uav_model.keras",
        "/kaggle/working/checkpoints/best_uav_model.keras"
    ] + glob.glob("**/*uav_model*.keras", recursive=True) + glob.glob("../**/*uav_model*.keras", recursive=True)

    MODEL_CKPT = None
    for c in ckpt_candidates:
        if os.path.exists(c) and os.path.getsize(c) > 10 * 1024 * 1024:
            MODEL_CKPT = c
            break

    if MODEL_CKPT is None:
        raise FileNotFoundError("Khong tim thay checkpoint trong so best_uav_model.keras! Vui long kiem tra thu muc weights/ hoac huan luyen o Muc 5.")

    print(f">> Nap Checkpoint: {MODEL_CKPT} ({os.path.getsize(MODEL_CKPT)/(1024*1024):.2f} MB)")
    try:
        infer_model = tf.keras.models.load_model(MODEL_CKPT, custom_objects={'CoordinateAttention': CoordinateAttention, 'AnchorFreeCenterHead': AnchorFreeCenterHead}, compile=False)
    except Exception:
        infer_model = build_detection_model(input_shape=Config.INPUT_SHAPE, fpn_dim=Config.FPN_DIM)
        infer_model.load_weights(MODEL_CKPT)

# 2. Xu ly tai tep neu la duong dan URL tu xa
source_path = MEDIA_SOURCE
if source_path.startswith("http://") or source_path.startswith("https://"):
    temp_dir = os.path.join(tempfile.gettempdir(), "anti_uav_inference")
    os.makedirs(temp_dir, exist_ok=True)
    fname = os.path.basename(source_path.split("?")[0]) or "downloaded_input.mp4"
    local_target = os.path.join(temp_dir, fname)
    if not os.path.exists(local_target):
        print(f">> Dang tai tep tu URL: {source_path}...")
        urllib.request.urlretrieve(source_path, local_target)
        print(f">> Da tai ve: {local_target}")
    source_path = local_target

# 3. Kiem tra dinh dang dau vao: Anh hay Video
is_image = any(source_path.lower().endswith(ext) for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"])

if is_image:
    print(f">> Xu ly suy luan anh: {source_path}")
    raw_bgr = cv2.imread(source_path)
    h_orig, w_orig = raw_bgr.shape[:2]
    rgb_norm = cv2.resize(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0
    t_in = np.expand_dims(np.concatenate([rgb_norm, rgb_norm, rgb_norm], axis=-1), axis=0)

    preds = infer_model(t_in, training=False)
    scores, bboxes = decode_detections(preds["heatmap"], preds["offset"], preds["size"])
    scores, bboxes = scores.numpy()[0], bboxes.numpy()[0]

    mask = scores >= CONFIDENCE_THRESHOLD
    scores, bboxes = scores[mask], bboxes[mask]

    tracker = TrajectoryEMAFilter(alpha=0.6, max_misses=5)
    disp = raw_bgr.copy()
    status = "SEARCHING"
    if len(scores) > 0:
        idx = np.argmax(scores)
        box = bboxes[idx]
        sc = scores[idx]
        tb, smooth_c, vel, state = tracker.update(box, sc)
        x1 = int(round(tb[0] * w_orig)); y1 = int(round(tb[1] * h_orig))
        x2 = int(round(tb[2] * w_orig)); y2 = int(round(tb[3] * h_orig))
        draw_reticle(disp, x1, y1, x2, y2, sc, "DRONE", tracker.trajectory, w_orig, h_orig)
        status = state

    canvas = create_canvas(disp, title="ANTI-UAV RGB", state=status, frame_info=f"Resolution: {w_orig}x{h_orig}")
    out_img_path = "outputs/annotated_single_cam_rgb.jpg"
    os.makedirs("outputs", exist_ok=True)
    cv2.imwrite(out_img_path, canvas)
    print(f">> Da luu anh suy luan thanh cong: {out_img_path}")

    # Hien thi anh trong notebook
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6), dpi=150)
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("ANTI-UAV RGB Tactical HUD", fontsize=12, fontweight="bold")
    plt.show()

else:
    print(f">> Xu ly suy luan video: {source_path}")
    cap = cv2.VideoCapture(source_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Khong the mo video: {source_path}")

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1920
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if MAX_FRAMES_TO_RENDER:
        total_frames = min(total_frames, MAX_FRAMES_TO_RENDER) if total_frames > 0 else MAX_FRAMES_TO_RENDER

    out_w, out_h = 960, 540
    canvas_w = out_w
    canvas_h = out_h + 60
    os.makedirs("outputs", exist_ok=True)
    raw_avi = "outputs/temp_single_cam_render.avi"
    final_mp4 = "outputs/rendered_single_cam_rgb.mp4"

    writer = cv2.VideoWriter(raw_avi, cv2.VideoWriter_fourcc(*"MJPG"), orig_fps, (canvas_w, canvas_h))
    tracker = TrajectoryEMAFilter(alpha=0.6, max_misses=10)

    buf = []
    pbar = tqdm(total=total_frames, desc="Render Single Cam RGB")
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        buf.append(frame)
        if len(buf) > 3:
            buf.pop(0)

        # Tao triplet temporal 9 kenh
        if len(buf) == 1:
            b_tm1, b_t, b_tp1 = buf[0], buf[0], buf[0]
        elif len(buf) == 2:
            b_tm1, b_t, b_tp1 = buf[0], buf[1], buf[1]
        else:
            b_tm1, b_t, b_tp1 = buf[0], buf[1], buf[2]

        im1 = cv2.resize(cv2.cvtColor(b_tm1, cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0
        im2 = cv2.resize(cv2.cvtColor(b_t,   cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0
        im3 = cv2.resize(cv2.cvtColor(b_tp1, cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0

        t_in = np.expand_dims(np.concatenate([im1, im2, im3], axis=-1), axis=0)

        t_start = time.time()
        preds = infer_model(t_in, training=False)
        scores, bboxes = decode_detections(preds["heatmap"], preds["offset"], preds["size"])
        infer_time = time.time() - t_start

        scores = scores.numpy()[0]
        bboxes = bboxes.numpy()[0]

        mask = scores >= CONFIDENCE_THRESHOLD
        scores = scores[mask]
        bboxes = bboxes[mask]

        disp_v = b_t.copy()
        state = "SEARCHING"
        spd = 0.0

        if len(scores) > 0:
            idx = np.argmax(scores)
            best_box = bboxes[idx]
            best_sc = scores[idx]
            smoothed_box, smooth_c, vel, state = tracker.update(best_box, best_sc)
            x1 = int(round(smoothed_box[0] * orig_w)); y1 = int(round(smoothed_box[1] * orig_h))
            x2 = int(round(smoothed_box[2] * orig_w)); y2 = int(round(smoothed_box[3] * orig_h))
            draw_reticle(disp_v, x1, y1, x2, y2, best_sc, "DRONE", tracker.trajectory, orig_w, orig_h)
            spd = np.sqrt(vel[0]**2 + vel[1]**2)
        else:
            smoothed_box, smooth_c, vel, state = tracker.update(None, 0.0)
            if state == "OCCLUDED" and smoothed_box is not None:
                x1 = int(round(smoothed_box[0] * orig_w)); y1 = int(round(smoothed_box[1] * orig_h))
                x2 = int(round(smoothed_box[2] * orig_w)); y2 = int(round(smoothed_box[3] * orig_h))
                draw_reticle(disp_v, x1, y1, x2, y2, 0.40, "OCCLUDED", tracker.trajectory, orig_w, orig_h)

        fps = 1.0 / max(infer_time, 1e-4)
        info_str = f"Frame: {frame_idx+1:04d}/{total_frames} | FPS: {fps:.1f} | Speed: {spd:.1f} px/f"
        canvas = create_canvas(disp_v, title="ANTI-UAV RGB", state=state, frame_info=info_str)

        writer.write(canvas)
        frame_idx += 1
        pbar.update(1)

        if MAX_FRAMES_TO_RENDER and frame_idx >= MAX_FRAMES_TO_RENDER:
            break

    pbar.close()
    cap.release()
    writer.release()

    # Chuyen doi sang dinh dang MP4 H.264 neu co san ffmpeg
    import shutil
    has_ffmpeg = shutil.which("ffmpeg") is not None
    converted = False
    if has_ffmpeg:
        cmd = f'ffmpeg -y -i "{raw_avi}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{final_mp4}" -loglevel error'
        ret = os.system(cmd)
        if ret == 0 and os.path.exists(final_mp4):
            converted = True
            try: os.remove(raw_avi)
            except Exception: pass
            print(f">> Video H.264 da san sang: {final_mp4}")

    if not converted:
        if os.path.exists(raw_avi):
            if os.path.exists(final_mp4): os.remove(final_mp4)
            os.rename(raw_avi, final_mp4)
        print(f">> Video da luu tai: {final_mp4}")

    # Phat video truc tiep tren Jupyter Notebook qua the HTML5
    try:
        mp4_bytes = open(final_mp4, "rb").read()
        data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
        display(HTML(f'''
        <div style="text-align: center; margin: 15px 0;">
            <h4 style="color: #00ffff; font-family: sans-serif;">ANTI-UAV RGB</h4>
            <video width="720" controls autoplay loop style="border: 2px solid #00ffff; border-radius: 8px;">
                <source src="{data_url}" type="video/mp4">
                Trinh duyet khong ho tro the video HTML5.
            </video>
        </div>
        '''))
    except Exception as e:
        print(f">> Khong the hien thi inline HTML5: {e}. Vui long xem file tai {final_mp4}")